# Лабораторная работа: скалярное автоматическое дифференцирование и обучение одного нейрона

Этот notebook входит в согласованную пару student/teacher. Полный код, математические объяснения, диагностические шаги и порядок заполнения находятся в [lab_guide.md](./lab_guide.md). Теоретическая часть находится в [theory.md](./theory.md). Выполняйте ячейки сверху вниз.


In [1]:
# AUTOGRAD_ESSENTIALS_CELL: imports_and_helper
import math
import random

random.seed(42)

def _check_close(a, b, tol=1e-6):
    """Мягкая проверка близости двух чисел."""
    assert abs(a - b) <= tol, f"{a} != {b} (tol={tol})"

def _central_diff(f, x, h=1e-6):
    """Центральная разность для численной производной."""
    return (f(x + h) - f(x - h)) / (2 * h)

print("imports and helpers ready")

imports and helpers ready


## Подготовка

Сначала перенесите код ячейки `imports_and_helper` из guide.


## Часть первая: класс Value

Заполняйте ячейку `value` постепенно по разделам guide. После каждой правки перезапускайте её и следующий относящийся к этапу эксперимент.


In [2]:
# AUTOGRAD_ESSENTIALS_CELL: value
class Value:
    """Скалярное значение с поддержкой автоматического дифференцирования."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    # ---------- служебные операции ----------
    def _coerce(self, other):
        return other if isinstance(other, Value) else Value(other)

    # ---------- прямой проход ----------
    def __add__(self, other):
        other = self._coerce(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = self._coerce(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-self._coerce(other))

    def __pow__(self, power):
        # поддерживаем только числовой (скалярный) показатель степени
        assert isinstance(power, (int, float)), "поддерживается только числовой power"
        out = Value(self.data ** power, (self,), f"**{power}")

        def _backward():
            self.grad += power * (self.data ** (power - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * (self._coerce(other) ** -1)

    # ---------- элементарные функции ----------
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")

        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")

        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(math.sin(self.data), (self,), "sin")

        def _backward():
            self.grad += math.cos(self.data) * out.grad
        out._backward = _backward
        return out

    def cos(self):
        out = Value(math.cos(self.data), (self,), "cos")

        def _backward():
            self.grad += -math.sin(self.data) * out.grad
        out._backward = _backward
        return out

    # ---------- обратный проход ----------
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    # ---------- вспомогательное ----------
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## Прямой проход: сложение


In [3]:
# AUTOGRAD_ESSENTIALS_CELL: addition_experiment
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a + b
e = d + c
e.backward()

print("d =", d, "e =", e)
print("grad a =", a.grad, "grad b =", b.grad, "grad c =", c.grad)
_check_close(a.grad, 1.0)
_check_close(b.grad, 1.0)
_check_close(c.grad, 1.0)

d = Value(data=-1.0000, grad=1.0000) e = Value(data=9.0000, grad=1.0000)
grad a = 1.0 grad b = 1.0 grad c = 1.0


## Прямой проход: умножение


In [4]:
# AUTOGRAD_ESSENTIALS_CELL: multiplication_experiment
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d * c
e.backward()

print("d =", d, "e =", e)
print("grad a =", a.grad, "grad b =", b.grad, "grad c =", c.grad)
_check_close(a.grad, -30.0)   # b*c
_check_close(b.grad, 20.0)    # a*c
_check_close(c.grad, -6.0)    # a*b

# Проверка: grad для a*a + a при a=2 равен 5
a = Value(2.0)
y = a * a + a
y.backward()
print("grad a*a+a при a=2:", a.grad)
_check_close(a.grad, 5.0)

d = Value(data=-6.0000, grad=10.0000) e = Value(data=-60.0000, grad=1.0000)
grad a = -30.0 grad b = 20.0 grad c = -6.0
grad a*a+a при a=2: 5.0


## Прямой проход: отрицание, вычитание и степень


In [5]:
# AUTOGRAD_ESSENTIALS_CELL: neg_sub_power_experiment
a = Value(3.0)
b = Value(2.0)

# отрицание
n = -a
n.backward()
print("grad (-a) по a:", a.grad)
_check_close(a.grad, -1.0)

# вычитание
a = Value(3.0); b = Value(2.0)
s = a - b
s.backward()
print("grad (a-b) по a, b:", a.grad, b.grad)
_check_close(a.grad, 1.0)
_check_close(b.grad, -1.0)

# степень
a = Value(3.0)
p = a ** 2
p.backward()
print("grad (a^2) по a:", a.grad)
_check_close(a.grad, 6.0)

a = Value(2.0)
p = a ** 3
p.backward()
_check_close(a.grad, 12.0)

# деление
a = Value(6.0); b = Value(3.0)
q = a / b
q.backward()
print("grad (a/b) по a, b:", a.grad, b.grad)
_check_close(a.grad, 1.0 / 3.0)
_check_close(b.grad, -6.0 / 9.0)

grad (-a) по a: -1.0
grad (a-b) по a, b: 1.0 -1.0
grad (a^2) по a: 6.0
grad (a/b) по a, b: 0.3333333333333333 -0.6666666666666666


## Прямой проход: элементарные функции


In [6]:
# AUTOGRAD_ESSENTIALS_CELL: elementary_experiment
# tanh
a = Value(0.5)
t = a.tanh()
t.backward()
print("tanh:", t.data, "grad:", a.grad)
_check_close(a.grad, 1 - math.tanh(0.5) ** 2)

# exp
a = Value(1.0)
e = a.exp()
e.backward()
_check_close(a.grad, math.exp(1.0))

# log
a = Value(2.0)
l = a.log()
l.backward()
_check_close(a.grad, 0.5)

# sin
a = Value(0.3)
s = a.sin()
s.backward()
_check_close(a.grad, math.cos(0.3))

# cos
a = Value(0.3)
c = a.cos()
c.backward()
_check_close(a.grad, -math.sin(0.3))

print("elementary functions ok")

tanh: 0.46211715726000974 grad: 0.7864477329659274
elementary functions ok


## Обратный проход и общий узел графа


In [7]:
# AUTOGRAD_ESSENTIALS_CELL: backward_shared_graph
a = Value(2.0)
b = Value(3.0)

# Общий узел: a используется дважды
c = a * b
d = a + b
e = c * d
e.backward()

# e = (a*b)*(a+b)
# de/da = b*(a+b) + (a*b) = 3*5 + 6 = 21
# de/db = a*(a+b) + (a*b) = 2*5 + 6 = 16
print("grad a =", a.grad, "grad b =", b.grad)
_check_close(a.grad, 21.0)
_check_close(b.grad, 16.0)

grad a = 21.0 grad b = 16.0


## Ручная трассировка локального обратного прохода

Эта ячейка делает отдельные локальные шаги графа $J = \tanh(ab+a)$ наблюдаемыми. Порядок, формулы и интерпретация приведены в [lab_guide.md](./lab_guide.md).


In [8]:
# AUTOGRAD_ESSENTIALS_CELL: manual_backward_trace
# J = tanh(a*b + a), a=1.0, b=2.0
a = Value(1.0)
b = Value(2.0)

ab = a * b          # u1 = a*b
s = ab + a          # u2 = u1 + a
J = s.tanh()        # J = tanh(u2)

# локальные производные
# dJ/du2 = 1 - tanh(u2)^2
# du2/du1 = 1, du2/da = 1
# du1/da = b, du1/db = a

# ручной обратный проход
dJ_du2 = 1 - math.tanh(s.data) ** 2
dJ_du1 = dJ_du2 * 1.0
dJ_da = dJ_du2 * 1.0 + dJ_du1 * b.data
dJ_db = dJ_du1 * a.data

J.backward()

print(f"J={J.data:.6f}")
print(f"grad a: autograd={a.grad:.6f}, manual={dJ_da:.6f}")
print(f"grad b: autograd={b.grad:.6f}, manual={dJ_db:.6f}")
_check_close(a.grad, dJ_da)
_check_close(b.grad, dJ_db)

J=0.995055
grad a: autograd=0.029598, manual=0.029598
grad b: autograd=0.009866, manual=0.009866


## Численная проверка центральной разностью


In [9]:
# AUTOGRAD_ESSENTIALS_CELL: finite_difference
def f_ab(av, bv):
    return math.tanh(av * bv + av)

a_val, b_val = 1.0, 2.0

# аналитический градиент через autograd
a = Value(a_val)
b = Value(b_val)
J = (a * b + a).tanh()
J.backward()

# численный градиент
num_da = _central_diff(lambda x: f_ab(x, b_val), a_val)
num_db = _central_diff(lambda x: f_ab(a_val, x), b_val)

print(f"da: autograd={a.grad:.8f}, numeric={num_da:.8f}")
print(f"db: autograd={b.grad:.8f}, numeric={num_db:.8f}")
_check_close(a.grad, num_da, tol=1e-4)
_check_close(b.grad, num_db, tol=1e-4)

da: autograd=0.02959811, numeric=0.02959811
db: autograd=0.00986604, numeric=0.00986604


## Часть вторая: один нейрон с гиперболическим тангенсом


In [10]:
# AUTOGRAD_ESSENTIALS_CELL: tanh_neuron
class TanhNeuron:
    """Один нейрон: y = tanh(w*x + b)."""

    def __init__(self, n_inputs=1, seed=None):
        rng = random.Random(seed) if seed is not None else random
        self.w = [Value(rng.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(rng.uniform(-1, 1))

    def __call__(self, x):
        # x — список чисел той же длины, что и self.w
        z = self.b
        for wi, xi in zip(self.w, x):
            z = z + wi * xi
        return z.tanh()

    def parameters(self):
        return self.w + [self.b]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

## Прямой проход нейрона


In [11]:
# AUTOGRAD_ESSENTIALS_CELL: neuron_forward_experiment
neuron = TanhNeuron(n_inputs=1, seed=0)
x = [1.0]
y = neuron(x)
print("w =", neuron.w, "b =", neuron.b)
print("выход нейрона:", y.data)
assert -1.0 <= y.data <= 1.0

w = [Value(data=0.6888, grad=0.0000)] b = Value(data=0.5159, grad=0.0000)
выход нейрона: 0.8350984858205053


## Среднеквадратичная ошибка и проверка градиентов параметров

В формуле средней потери $i$ задаёт фиксированный объект, а $\mu$ — немой индекс суммы по всем объектам. В Python эту сумму строит `zip(inputs, targets)`, поэтому переменная с именем `mu` не нужна.


In [12]:
# AUTOGRAD_ESSENTIALS_CELL: loss_and_gradient
def mse_loss(neuron, inputs, targets):
    """Средняя квадратичная ошибка по всем объектам."""
    total = None
    n = len(inputs)
    for x, t in zip(inputs, targets):
        pred = neuron(x)
        diff = pred - t
        term = diff * diff
        total = term if total is None else total + term
    return total * (1.0 / n)

# демонстрационный датасет: y = x^2 на нескольких точках (нелинейная задача)
inputs = [[-2.0], [-1.0], [0.0], [1.0], [2.0]]
targets = [4.0, 1.0, 0.0, 1.0, 4.0]

# ограничимся нормировкой, чтобы tanh-нейрон мог выучить сигнал
inputs = [[x[0] / 2.0] for x in inputs]
targets = [t / 4.0 for t in targets]

neuron = TanhNeuron(n_inputs=1, seed=1)
loss = mse_loss(neuron, inputs, targets)
neuron.zero_grad()
loss.backward()

print(f"loss = {loss.data:.6f}")
for i, p in enumerate(neuron.parameters()):
    print(f"param {i}: data={p.data:.6f}, grad={p.grad:.6f}")

# проверка градиентов центральной разностью
def loss_value(neuron, inputs, targets):
    return mse_loss(neuron, inputs, targets).data

h = 1e-6
for i, p in enumerate(neuron.parameters()):
    old = p.data
    p.data = old + h
    lp = loss_value(neuron, inputs, targets)
    p.data = old - h
    lm = loss_value(neuron, inputs, targets)
    p.data = old
    num = (lp - lm) / (2 * h)
    print(f"param {i}: analytic={p.grad:.6f}, numeric={num:.6f}")
    _check_close(p.grad, num, tol=1e-4)

loss = 0.347815
param 0: data=-0.731272, grad=-0.433789
param 1: data=0.694867, grad=-0.163087
param 0: analytic=-0.433789, numeric=-0.433789
param 1: analytic=-0.163087, numeric=-0.163087


## Градиентный спуск и диагностика обучения


In [13]:
# AUTOGRAD_ESSENTIALS_CELL: train_neuron
def train_neuron(neuron, inputs, targets, epochs=500, lr=0.1, log_every=100):
    history = []
    for epoch in range(epochs):
        # forward + backward
        neuron.zero_grad()
        loss = mse_loss(neuron, inputs, targets)
        loss.backward()

        # шаг градиентного спуска
        for p in neuron.parameters():
            p.data -= lr * p.grad

        history.append(loss.data)
        if epoch % log_every == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:4d}: loss = {loss.data:.6f}")

    return history

# сброс seed для воспроизводимости
random.seed(42)
neuron = TanhNeuron(n_inputs=1, seed=7)
history = train_neuron(neuron, inputs, targets, epochs=500, lr=0.5, log_every=100)

final_loss = mse_loss(neuron, inputs, targets).data
print(f"Финальная ошибка: {final_loss:.6f}")

# точность: считаем, что предсказание верно, если знак совпадает
correct = 0
for x, t in zip(inputs, targets):
    pred = neuron(x).data
    if (pred >= 0) == (t >= 0):
        correct += 1
accuracy = correct / len(inputs)
print(f"Точность: {accuracy:.2f}")
print("w =", [p.data for p in neuron.w], "b =", neuron.b.data)

epoch    0: loss = 1.351476
epoch  100: loss = 0.175000
epoch  200: loss = 0.175000
epoch  300: loss = 0.175000
epoch  400: loss = 0.175000
epoch  499: loss = 0.175000
Финальная ошибка: 0.175000
Точность: 1.00
w = [-3.469446951953614e-17] b = 0.5493061443340549


## Чек-лист

- [ ] Пройдены все проверки из guide.
- [ ] Градиент для `a * a + a` равен 5 при `a = 2`.
- [ ] Численный и аналитический градиенты близки.
- [ ] Финальная ошибка меньше 0.03, точность равна 1.

Полный чек-лист, диагностика и вопросы находятся в конце [lab_guide.md](./lab_guide.md).
